In [1]:
!git clone -q https://github.com/smgroves/demand_lab.git

%cd /content/demand_lab

from demand import demand

responses = demand(price=14.0)

print(len(responses), responses.sum())

/content/demand_lab
202 166


In [2]:

import numpy as np

# Test the two prices
responses_30 = demand(price=30)
responses_25 = demand(price=25)

# Proportion of customers who buy
p_30 = responses_30.mean()
p_25 = responses_25.mean()

# Revenue per potential customer
r_30 = 30 * p_30
r_25 = 25 * p_25

print(f"$30: n={len(responses_30)}, p_hat={p_30:.3f}, r_hat=${r_30:.2f}")
print(f"$25: n={len(responses_25)}, p_hat={p_25:.3f}, r_hat=${r_25:.2f}")



$30: n=215, p_hat=0.381, r_hat=$11.44
$25: n=155, p_hat=0.439, r_hat=$10.97


In [3]:

rng = np.random.default_rng(5030)
R = 5000

def bootstrap_revenue(responses, price):
    n = len(responses)
    estimates = []

    for i in range(R):
        sample = rng.choice(responses, size=n, replace=True)
        revenue = price * sample.mean()
        estimates.append(revenue)

    return np.array(estimates)

boot_30 = bootstrap_revenue(responses_30, 30)
boot_25 = bootstrap_revenue(responses_25, 25)

se_30 = np.std(boot_30, ddof=1)
se_25 = np.std(boot_25, ddof=1)

print(f"SE at $30: ${se_30:.3f}")
print(f"SE at $25: ${se_25:.3f}")


SE at $30: $0.997
SE at $25: $0.999


In [4]:

# 95% Normal confidence intervals
ci_30 = (r_30 - 1.96 * se_30, r_30 + 1.96 * se_30)
ci_25 = (r_25 - 1.96 * se_25, r_25 + 1.96 * se_25)

# Difference in estimated revenue
diff = r_25 - r_30

# Independent-group standard error
se_diff = np.sqrt(se_25**2 + se_30**2)

# 95% CI for the difference
ci_diff = (
    diff - 1.96 * se_diff,
    diff + 1.96 * se_diff
)

print("95% CI at $30:", ci_30)
print("95% CI at $25:", ci_25)
print("Estimated difference:", diff)
print("SE of difference:", se_diff)
print("95% CI of difference:", ci_diff)


95% CI at $30: (np.float64(9.487799012668953), np.float64(13.395921917563607))
95% CI at $25: (np.float64(9.009279216465446), np.float64(12.926204654502298))
Estimated difference: -0.47411852963240797
SE of difference: 1.4115174100325028
95% CI of difference: (np.float64(-3.2406926532961133), np.float64(2.2924555940312974))


## 1. Compare two prices
   
5. Interpret the estimates, SEs, and intervals. Report which price earns more profit, and how confident you are.

I calculated the estimated revenues per potental customers based on first market tests which is `$11.44` at a price of `$30` and `$10.97` at a price of `$25`. The difference of them is approximately `$0.47`, so the `$30` price generate more revenue. Then, I bootstraped, and resample it 5000 times to check if that difference is in estimated revenue is meaningful or could be due to sampling variability. The standard errors after bootstrap for `$30` price is `$0.997`, and `$0.999` for $25 price which shows uncertainty in our estimates. Also, the `95%` confidence interval revenue difference `($25 - $30)` is `(-$3.24, $2.29)`. Since, the confidence interval includes zero, we fail to reject the null hypthesis that two prices have equal expected revenue.  


In [5]:

# Second market test at each price
responses_30_second = demand(price=30)
responses_25_second = demand(price=25)

# Combine first and second samples for each price
pooled_30 = np.concatenate([responses_30, responses_30_second])
pooled_25 = np.concatenate([responses_25, responses_25_second])

print("Total observations at $30:", len(pooled_30))
print("Total observations at $25:", len(pooled_25))

print("Updated revenue at $30:", 30 * pooled_30.mean())
print("Updated revenue at $25:", 25 * pooled_25.mean())


Total observations at $30: 442
Total observations at $25: 327
Updated revenue at $30: 12.081447963800905
Updated revenue at $25: 10.244648318042813


In [6]:

# Bootstrap the combined samples
boot_30_pooled = bootstrap_revenue(pooled_30, 30)
boot_25_pooled = bootstrap_revenue(pooled_25, 25)

# Updated standard errors
se_30_new = np.std(boot_30_pooled, ddof=1)
se_25_new = np.std(boot_25_pooled, ddof=1)

# Updated estimated revenues
r_30_new = 30 * pooled_30.mean()
r_25_new = 25 * pooled_25.mean()

# Updated 95% confidence intervals
ci_30_new = (
    r_30_new - 1.96 * se_30_new,
    r_30_new + 1.96 * se_30_new
)

ci_25_new = (
    r_25_new - 1.96 * se_25_new,
    r_25_new + 1.96 * se_25_new
)

# Revenue difference and its standard error
diff_new = r_25_new - r_30_new
se_diff_new = np.sqrt(se_25_new**2 + se_30_new**2)

# 95% CI for the difference
ci_diff_new = (
    diff_new - 1.96 * se_diff_new,
    diff_new + 1.96 * se_diff_new
)

print(f"New SE at $30: ${se_30_new:.3f}")
print(f"New SE at $25: ${se_25_new:.3f}")

print(f"95% CI at $30: {ci_30_new}")
print(f"95% CI at $25: {ci_25_new}")
print(f"95% CI for difference: {ci_diff_new}")


New SE at $30: $0.699
New SE at $25: $0.677
95% CI at $30: (np.float64(10.712108183172814), np.float64(13.450787744428995))
95% CI at $25: (np.float64(8.917791216106888), np.float64(11.571505419978738))
95% CI for difference: (np.float64(-3.7435353914156777), np.float64(0.06993609989949467))


## 2. The cost of sampling

8. Recompute the SEs, the intervals, and the interval for the difference. By roughly what factor did the SE shrink? Explain why, using the SE formula.

$$
SE(\bar X)=\frac{s}{\sqrt{n}}
$$

for `$30` the Standard error decreased roughly by 30%, since we have more observations:

$$
\frac{0.997 - 0.699}{0.997} \times 100 \approx 29.9\%
$$

for `$25` the Standard error decreased roughly by 32%, since we have more observations:

$$
\frac{0.999 - 0.677}{0.999} \times 100 \approx 32.2\%
$$

   
9.  **Was the extra sampling worth it?** Every market test takes time, and while you're testing you aren't selling at your best price, so each test costs the company revenue. Did the first round already settle the question (its difference interval excluded 0)? If so, the second round cost time and bought nothing but a narrower interval. If not, did the second round settle it?

The additional sampling increased the revenue for $30 price up to `$12.08`, and teh estimated revenue decreased to `$10.24`, and their difference is about `-$1.84`. The `95% CI` for the difference is `(−$3.74, $0.07)` which got narrower. But, it still include zero, and we fail to reject the null hypothesis.

Increasing the sampling was usefull and helped us to reduce the standard error by approxmitlly `30%` which give us more precise estimates for revenues. In addition, it supported the evidance in favor of `$30` price from the first round.

However, as I mentioned erlier the `95% CI` still includes zero and we cannot confidently tell which price will expect the company to have more revenue. So, whetther trying the second round worth it depends on how valuable the new improved information is for teh company.

In [7]:

def analyze_price(responses, price, R=5000):
    # Number of customers
    n = len(responses)

    # Purchase proportion
    p_hat = responses.mean()

    # Estimated revenue per potential customer
    r_hat = price * p_hat

    # Bootstrap revenue estimates
    boot = bootstrap_revenue(responses, price)

    # Bootstrap standard error
    se = np.std(boot, ddof=1)

    # 95% confidence interval
    ci_low = r_hat - 1.96 * se
    ci_high = r_hat + 1.96 * se

    return {
        "price": price,
        "n": n,
        "p_hat": p_hat,
        "revenue": r_hat,
        "se": se,
        "ci_low": ci_low,
        "ci_high": ci_high
    }


In [8]:

# Store our existing results
results = []

results.append(analyze_price(pooled_25, 25))
results.append(analyze_price(pooled_30, 30))

# Display results
import pandas as pd

pd.DataFrame(results).round(3)


,price,n,p_hat,revenue,se,ci_low,ci_high
0,25,327,0.410,10.245,0.686,8.900,11.590
1,30,442,0.403,12.081,0.686,10.737,13.426


In [9]:

# New market test at $40
responses_40 = demand(price=40)

# Analyze the results
result_40 = analyze_price(responses_40, 40)

# Save the result
results.append(result_40)

# Compare all prices tested so far
pd.DataFrame(results).round(3)


,price,n,p_hat,revenue,se,ci_low,ci_high
0,25,327,0.410,10.245,0.686,8.900,11.590
1,30,442,0.403,12.081,0.686,10.737,13.426
2,40,166,0.380,15.181,1.497,12.246,18.115


In [10]:

# New market test at $60
responses_60 = demand(price=60)

# Analyze the results
result_60 = analyze_price(responses_60, 60)

# Save the result
results.append(result_60)

# Compare all tested prices
pd.DataFrame(results).round(3)


,price,n,p_hat,revenue,se,ci_low,ci_high
0,25,327,0.410,10.245,0.686,8.900,11.590
1,30,442,0.403,12.081,0.686,10.737,13.426
2,40,166,0.380,15.181,1.497,12.246,18.115
3,60,147,0.286,17.143,2.250,12.733,21.553


In [11]:

# Test a higher price: $80
responses_80 = demand(price=80)

# Analyze the market test
result_80 = analyze_price(responses_80, 80)

# Save the results
results.append(result_80)

# Compare all prices tested
pd.DataFrame(results).round(3)


,price,n,p_hat,revenue,se,ci_low,ci_high
0,25,327,0.410,10.245,0.686,8.900,11.590
1,30,442,0.403,12.081,0.686,10.737,13.426
2,40,166,0.380,15.181,1.497,12.246,18.115
3,60,147,0.286,17.143,2.250,12.733,21.553
4,80,163,0.178,14.233,2.393,9.543,18.923


In [12]:

# Hill climbing: test $50
responses_50 = demand(price=50)

# Analyze the result
result_50 = analyze_price(responses_50, 50)

# Save the result
results.append(result_50)

# Sort by price for easier comparison
pd.DataFrame(results).sort_values("price").round(3)


,price,n,p_hat,revenue,se,ci_low,ci_high
0,25,327,0.410,10.245,0.686,8.900,11.590
1,30,442,0.403,12.081,0.686,10.737,13.426
2,40,166,0.380,15.181,1.497,12.246,18.115
5,50,203,0.305,15.271,1.635,12.066,18.476
3,60,147,0.286,17.143,2.250,12.733,21.553
4,80,163,0.178,14.233,2.393,9.543,18.923


In [13]:

# Hill climbing: test $70
responses_70 = demand(price=70)

# Analyze the result
result_70 = analyze_price(responses_70, 70)

# Save the result
results.append(result_70)

# Compare all prices
pd.DataFrame(results).sort_values("price").round(3)


,price,n,p_hat,revenue,se,ci_low,ci_high
0,25,327,0.410,10.245,0.686,8.900,11.590
1,30,442,0.403,12.081,0.686,10.737,13.426
2,40,166,0.380,15.181,1.497,12.246,18.115
5,50,203,0.305,15.271,1.635,12.066,18.476
3,60,147,0.286,17.143,2.250,12.733,21.553
6,70,142,0.296,20.704,2.691,15.429,25.979
4,80,163,0.178,14.233,2.393,9.543,18.923


In [15]:

# Hill climbing: test $65
responses_65 = demand(price=65)

result_65 = analyze_price(responses_65, 65)

results.append(result_65)

pd.DataFrame(results).sort_values("price").round(3)


,price,n,p_hat,revenue,se,ci_low,ci_high
0,25,327,0.410,10.245,0.686,8.900,11.590
1,30,442,0.403,12.081,0.686,10.737,13.426
2,40,166,0.380,15.181,1.497,12.246,18.115
5,50,203,0.305,15.271,1.635,12.066,18.476
3,60,147,0.286,17.143,2.250,12.733,21.553
7,65,136,0.235,15.294,2.381,10.627,19.961
6,70,142,0.296,20.704,2.691,15.429,25.979
4,80,163,0.178,14.233,2.393,9.543,18.923


In [16]:

# Hill climbing: test $75
responses_75 = demand(price=75)

# Analyze the result
result_75 = analyze_price(responses_75, 75)

# Save the result
results.append(result_75)

# Compare all tested prices
pd.DataFrame(results).sort_values("price").round(3)


,price,n,p_hat,revenue,se,ci_low,ci_high
0,25,327,0.410,10.245,0.686,8.900,11.590
1,30,442,0.403,12.081,0.686,10.737,13.426
2,40,166,0.380,15.181,1.497,12.246,18.115
5,50,203,0.305,15.271,1.635,12.066,18.476
3,60,147,0.286,17.143,2.250,12.733,21.553
7,65,136,0.235,15.294,2.381,10.627,19.961
6,70,142,0.296,20.704,2.691,15.429,25.979
8,75,205,0.283,21.220,2.355,16.603,25.836
4,80,163,0.178,14.233,2.393,9.543,18.923


In [23]:

# Create a table for revenue differences compared with $75
df = pd.DataFrame(results)

# Reference price: $75
reference_price = 75
reference = df[df["price"] == reference_price].iloc[0]

diff_results = []

for _, row in df.iterrows():

    # Skip comparing $75 with itself
    if row["price"] == reference_price:
        continue

    # Revenue difference: $75 minus the other price
    diff = reference["revenue"] - row["revenue"]

    # Standard error of the difference (independent samples)
    se_diff = np.sqrt(reference["se"]**2 + row["se"]**2)

    # 95% confidence interval for the difference
    ci_low = diff - 1.96 * se_diff
    ci_high = diff + 1.96 * se_diff

    # Check whether the interval includes zero
    includes_zero = ci_low <= 0 <= ci_high

    # Save comparison results
    diff_results.append({
        "Comparison": f"$75 - ${int(row['price'])}",
        "Difference": diff,
        "SE_diff": se_diff,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "Includes_zero": includes_zero
    })

# Create a separate DataFrame
diff_df = pd.DataFrame(diff_results)

# Sort comparisons by the price being compared
diff_df["Other_price"] = (
    diff_df["Comparison"].str.split("$").str[-1].astype(int)
)

diff_df = (
    diff_df
    .sort_values("Other_price")
    .drop(columns="Other_price")
    .reset_index(drop=True)
)

# Display the final table
diff_df.round(3)


,Comparison,Difference,SE_diff,CI_low,CI_high,Includes_zero
0,$75 - $25,10.975,2.453,6.167,15.783,False
1,$75 - $30,9.138,2.453,4.330,13.946,False
2,$75 - $40,6.039,2.791,0.569,11.509,False
3,$75 - $50,5.949,2.867,0.329,11.569,False
4,$75 - $60,4.077,3.257,-2.308,10.461,True
5,$75 - $65,5.925,3.349,-0.639,12.490,True
6,$75 - $70,0.515,3.576,-6.494,7.525,True
7,$75 - $80,6.986,3.358,0.405,13.567,False


In [24]:

# Second market test at $70 and $75
responses_70_second = demand(price=70)
responses_75_second = demand(price=75)

# Combine observations
pooled_70 = np.concatenate([responses_70, responses_70_second])
pooled_75 = np.concatenate([responses_75, responses_75_second])

# Recalculate estimates
result_70_new = analyze_price(pooled_70, 70)
result_75_new = analyze_price(pooled_75, 75)

# Replace previous estimates
results = [
    r for r in results
    if r["price"] not in [70, 75]
]

results.extend([result_70_new, result_75_new])

# Updated table
pd.DataFrame(results).sort_values("price").round(3)


,price,n,p_hat,revenue,se,ci_low,ci_high
0,25,327,0.410,10.245,0.686,8.900,11.590
1,30,442,0.403,12.081,0.686,10.737,13.426
2,40,166,0.380,15.181,1.497,12.246,18.115
5,50,203,0.305,15.271,1.635,12.066,18.476
3,60,147,0.286,17.143,2.250,12.733,21.553
6,65,136,0.235,15.294,2.381,10.627,19.961
7,70,356,0.270,18.876,1.642,15.659,22.094
8,75,369,0.279,20.935,1.767,17.471,24.399
4,80,163,0.178,14.233,2.393,9.543,18.923


First, I tried to test different prices in a wider range, as you saw we tried `$25` and `$30` which estimated that the company will have higher revenue. So, I tried to highere prices to chech how it affect the revenue, calculated the purchase proportion, estimated reneue, bootstrap standard error, and `95%` confidence interval using `5,000` bootstrap resamples.

I noticed that, estimated revenue increased from `$12.08 at $30`, to `$15.18 at $40`, and `$17.14 at $60`, but when i increased the price to `$80` teh revenue dropped to `$14.23`. This gave me the hint, and Ifocuded on preices around `$40-$80`, and test small price increment. Therefore, i tried `$50, $70, $75`, and among all of them `$75` got highest etimated revenue(`20.70`).

Altough we got the hiehst estimated revenue at price `$75`, I tried to consider SE and `95% CI` to better undrestand the uncertainty in my estimates. So, I creaate another table for the `95% CI for difference` to compare `$75` with other prices. I found out from result that we have statistically significant positive renenue compare to `25, 30, 40, 50 and 80`. However, `95% CI for the differences` between `$75 and $60, $65, 70` included zero which indicates that the company is epected to have higher revenue at price `$75` than `$75 and $60, $65, 70` becuase we don't have enough statistical evidance.

in teh next step, I tried to have additional sampling for `$70, $75`, and combined with previous samples.

**Here is the result after addiytional sampling:**


| Statistic | \$70 Price | \$75 Price |
|---|---:|---:|
| Total observations | 356 | 369 |
| Purchase proportion | 27.0% | 27.9% |
| Estimated revenue | \$18.88 | \$20.93 |
| Standard error | \$1.642 | \$1.767 |
| 95% confidence interval | (\$15.66, \$22.09) | (\$17.47, \$24.40) |

then I calculted the `95% CI differences` between 75 and 70 dollars.

$$
\hat r_{75}-\hat r_{70}=20.935-18.876=2.059
$$

**The standard error:**

$$
SE_{\text{diff}}=\sqrt{1.767^2+1.642^2}\approx2.412
$$

**95% confidence interval for the difference was:**

$$
2.059\pm1.96(2.412)\approx(-2.67,\ 6.79)
$$

The confidence interval still includes zero, and it means that we still don't have enough statistical evidence to conclude that `$75` will generate more expected revenue than `$70` at `5%` significance level. But additional sampling, helped us to reduce uncertainty and provide more precise estimates


## Final Decision:

Based on the market tests with different prices, I would choose `$75` as a best price, since we had the highest estimated revenue per potential customer, which was approximately `$20.93` after combining the samples. However, `$75` performed better in our observed data the confidence interval for its revenue difference compared with the `$70` still includes zero. So, we cannot claim with statistical confidence that `$75` is truly better than `$70`. And also, I consider that the cost of additional market tests will take times and can cost the company potential revenue. since I had nine prices and collected additional observations at two leading prices, I decided to stop here rather than continue spending more resources on uncertainty improvement. So overall, I found that the estimated revenue helped us identify our promising prices, but I also noticed that the standard error and confidence interval help me a lot to understand the uncertainty in our decisions. `So based on available evidence that I got from the test, I believe that $75 is my final recommended price, while I have to acknowledge that the true optimal price remains uncertain.`



### AI Assistance Disclosure

I used ChatGPT as a supporting tool during this lab, mainly to help debug Python code, improve Markdown formatting (including tables and mathematical formulas), and create reusable functions to test different prices and organize the results into a single comparison table.

The analysis, interpretation of the results, statistical reasoning, and final pricing decision were completed independently by me.